In [1]:
import os
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset
from huggingface_hub import hf_hub_download
# from unsloth import FastVisionModel, is_bfloat16_supported, UnslothVisionDataCollator
# from trl import SFTTrainer, SFTConfig
from huggingface_hub.utils import logging, disable_progress_bars

# # 2. Tắt hoàn toàn tất cả các thanh tiến trình (progress bar) của Hugging Face trên toàn hệ thống
# disable_progress_bars()
# logging.set_verbosity_error() 
# os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [7]:
from kaggle_secrets import UserSecretsClient
secret_label = "HUGGINGFACE_HUB_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)
import os
os.environ["HF_TOKEN"] = secret_value 

In [4]:
# ==========================================
# 1. CẤU HÌNH THAM SỐ
# ==========================================
HF_REPO_TEXT_ID = "UngLong/openm3chest-labels-v2"
HF_REPO_IMAGE_ID = "UngLong/openm3chest-npy-v2"

In [5]:
print("Loading dataset...")
dataset_1 = load_dataset(HF_REPO_TEXT_ID, split="train", name="CVD_diagnosis")
dataset_2 = load_dataset(HF_REPO_TEXT_ID, split="train", name="CVD_mortality")

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

CVD_diagnosis/train-00000-of-00001.parqu(…):   0%|          | 0.00/1.18M [00:00<?, ?B/s]

CVD_diagnosis/test-00000-of-00001.parque(…):   0%|          | 0.00/2.78M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3759 [00:00<?, ? examples/s]

CVD_mortality/train-00000-of-00001.parqu(…):   0%|          | 0.00/1.18M [00:00<?, ?B/s]

CVD_mortality/test-00000-of-00001.parque(…):   0%|          | 0.00/2.78M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3759 [00:00<?, ? examples/s]

In [12]:
dataset_1[0]

{'keys': '1.2.840.113654.2.55.36582369976962245491974432349710616714',
 'pids': '100088',
 'clinical_data': '{"demo": {"age": 70.0, "educat": "Associate degree/ some college", "ethnic": "Hispanic or Latino", "gender": "Female", "height": 62.0, "race": "White", "weight": 175.0}, "smoking": {"age_quit": 57.0, "cigar": "No", "cigsmok": "Former", "pipe": "No", "pkyr": 61.5, "smokeage": 16.0, "smokeday": 30.0, "smokelive": "Yes", "smokework": "Yes", "smokeyr": 41.0}, "disease_his": {}, "cancer_his": {}, "fam_lc": {}}',
 'questions': ['Is there any significant cardiovascular abnormality?',
  'Can you identify any notable abnormalities in the cardiovascular system?',
  'Are there any evident issues related to the cardiovascular system in the findings?',
  'Are there discernible cardiovascular defects present?',
  'Do you notice any cardiovascular anomalies of significance?',
  'Has any cardiovascular abnormality been detected?',
  'Are there any significantly unusual cardiovascular findings t

In [14]:
dataset_2[0]

{'keys': '1.2.840.113654.2.55.36582369976962245491974432349710616714',
 'pids': '100088',
 'clinical_data': '{"demo": {"age": 70.0, "educat": "Associate degree/ some college", "ethnic": "Hispanic or Latino", "gender": "Female", "height": 62.0, "race": "White", "weight": 175.0}, "smoking": {"age_quit": 57.0, "cigar": "No", "cigsmok": "Former", "pipe": "No", "pkyr": 61.5, "smokeage": 16.0, "smokeday": 30.0, "smokelive": "Yes", "smokework": "Yes", "smokeyr": 41.0}, "disease_his": {}, "cancer_his": {}, "fam_lc": {}}',
 'questions': ['Predict the risk of cardiovascular disease mortality.',
  'What is the risk of cardiovascular disease mortality?',
  'Can you assess the likelihood of mortality due to cardiovascular disease?',
  'How likely is death from cardiovascular disease?',
  'What are the chances of dying from cardiovascular disease?',
  'Could you estimate the mortality risk associated with cardiovascular disease?',
  'Can you determine the risk level for death caused by cardiovascula

# Luồng để xử lý và upload lên HF

In [31]:
import json
import pandas as pd
from datasets import load_dataset, Dataset
import random 

# 1. Load dataset từ Hugging Face
print("Loading dataset...")
dataset_1 = load_dataset(HF_REPO_TEXT_ID, split="train", name="CVD_diagnosis")
dataset_2 = load_dataset(HF_REPO_TEXT_ID, split="train", name="CVD_mortality")

# Chuyển sang Pandas DataFrame để xử lý gộp cho dễ
df1 = pd.DataFrame(dataset_1)
df2 = pd.DataFrame(dataset_2)

# 2. Hàm xử lý ĐỘNG clinical_data thành đoạn văn (Không lo thiếu/thừa trường)
def flatten_clinical_data(clinical_str):
    if not clinical_str:
        return "No clinical history available."
    
    try:
        data = json.loads(clinical_str)
    except Exception:
        return str(clinical_str)
    
    paragraphs = []
    # Đi qua từng nhóm lớn: demo, smoking, disease_his, v.v.
    for category, details in data.items():
        if not details: # Bỏ qua nếu dict trống như disease_his: {}
            continue
        
        # Biến đổi các key-value thành chuỗi, ví dụ: "age is 70.0"
        detail_strs = []
        for k, v in details.items():
            # Thay thế dấu gạch dưới cho dễ đọc nếu có
            clean_key = k.replace('_', ' ')
            detail_strs.append(f"{clean_key}: {v}")
            
        category_text = f"- {category.capitalize()}: {', '.join(detail_strs)}."
        paragraphs.append(category_text)
        
    return " Clinical data summary:\n" + "\n".join(paragraphs)

# 3. Gộp 2 dataset dựa trên pids và keys
# Đổi tên cột trước khi merge để phân biệt label và questions
df1 = df1.rename(columns={'labels': 'label_diag', 'questions': 'ques_diag', 'answer_dict': 'ans_dict_diag'})
df2 = df2.rename(columns={'labels': 'label_mort', 'questions': 'ques_mort', 'answer_dict': 'ans_dict_mort'})

# Lấy các cột chung và cột riêng cần thiết để merge
merged_df = pd.merge(
    df1[['keys', 'pids', 'clinical_data', 'ques_diag', 'label_diag', 'ans_dict_diag']], 
    df2[['keys', 'pids', 'ques_mort', 'label_mort', 'ans_dict_mort']], 
    on=['pids', 'keys'], 
    how='inner'
)

# 4. Xử lý tạo cột mới cho Ready Dataset
prompts = []
responses = []

for idx, row in merged_df.iterrows():
    # Chuyển clinical_data thành đoạn văn bản
    clinical_paragraph = flatten_clinical_data(row['clinical_data'])
    
    # Lấy ngẫu nhiên hoặc lấy câu hỏi đầu tiên của mỗi bên để ghép thành câu hỏi kép
    q_diag = random.choice(row['ques_diag'])
    q_mort = random.choice(row['ques_mort'])
    
    # Tạo câu hỏi tích hợp (Multi-task question)
    combined_question = (
        f"Based on the clinical data provided below, please answer the following two questions:\n"
        f"1. {q_diag}\n"
        f"2. {q_mort}"
    )
    
    # Tạo full Prompt đầu vào cho AI
    full_prompt = f"""You are an advanced Multimodal AI Clinical Decision Support System specializing in Cardiothoracic Imaging. Your task is to analyze the provided patient clinical record along with all corresponding Chest CT scan slices provided in the visual input to perform Diagnostic Screening and Mortality Risk Prediction.
    
    [PATIENT CLINICAL RECORD]
    {clinical_paragraph}
    
    [EVALUATION REQUEST]
    Based on the comprehensive and integrated analysis of the provided Chest CT scan slice(s) and the clinical record above, formulate clinical judgments to address the following inquiries:
    1. {q_diag}
    2. {q_mort}
    
    [OUTPUT FORMAT INSTRUCTION]
    You must respond exclusively in a structured XML format using the exact schema below. Do not include any introductory remarks, explanations, or conversational text. Provide your final answers as text descriptions, not numeric values.
    
    Expected Format:
    <prediction>
      <cardiovascular_abnormality>[Your textual answer here]</cardiovascular_abnormality>
      <mortality_risk>[Your textual answer here]</mortality_risk>
    </prediction>"""
    prompts.append(full_prompt)
    
    # Giải mã nhãn từ số sang chữ (Sử dụng answer_dict)
    dict_diag = json.loads(row['ans_dict_diag'])
    dict_mort = json.loads(row['ans_dict_mort'])
    
    text_diag = dict_diag.get(str(row['label_diag']), "Unknown")
    text_mort = dict_mort.get(str(row['label_mort']), "Unknown")
    
    # Tạo cấu trúc XML cho Output (Chuẩn hóa thụt lề và đóng thẻ chính xác)
    xml_response = (
        """<prediction>\n"""
        f"""  <cardiovascular_abnormality>{text_diag}</cardiovascular_abnormality>\n"""
        f"""  <mortality_risk>{text_mort}</mortality_risk>\n"""
        """</prediction>""")
        
    responses.append(xml_response)

# Thêm 2 cột kết quả vào DataFrame
merged_df['prompt'] = prompts
merged_df['response'] = responses

# 5. Chuyển ngược về Hugging Face Dataset nếu bạn muốn push up lại hoặc lưu local
final_dataset = Dataset.from_pandas(merged_df[['keys', 'pids', 'prompt', 'response']])

# Xem thử sample đầu tiên
print("\n--- SAMPLE PROMPT ---")
print(final_dataset[0]['prompt'])
print("\n--- SAMPLE RESPONSE (XML) ---")
print(final_dataset[0]['response'])

Loading dataset...

--- SAMPLE PROMPT ---
You are an advanced Multimodal AI Clinical Decision Support System specializing in Cardiothoracic Imaging. Your task is to analyze the provided patient clinical record along with all corresponding Chest CT scan slices provided in the visual input to perform Diagnostic Screening and Mortality Risk Prediction.
    
    [PATIENT CLINICAL RECORD]
     Clinical data summary:
- Demo: age: 70.0, educat: Associate degree/ some college, ethnic: Hispanic or Latino, gender: Female, height: 62.0, race: White, weight: 175.0.
- Smoking: age quit: 57.0, cigar: No, cigsmok: Former, pipe: No, pkyr: 61.5, smokeage: 16.0, smokeday: 30.0, smokelive: Yes, smokework: Yes, smokeyr: 41.0.
    
    [EVALUATION REQUEST]
    Based on the comprehensive and integrated analysis of the provided Chest CT scan slice(s) and the clinical record above, formulate clinical judgments to address the following inquiries:
    1. Do you notice any cardiovascular anomalies of significanc

In [32]:
# # ==========================================
# 3. UPLOAD LÊN HUGGING FACE HUB
# ==========================================
# Thay bằng tên tài khoản và tên repo bạn muốn tạo/up lên trên HF
HF_REPO_OUTPUT = "rimine/Cardiology_ready_dataset" 

print(f"Uploading dataset to Hugging Face Hub: {HF_REPO_OUTPUT}...")
final_dataset.push_to_hub(
    repo_id=HF_REPO_OUTPUT,
    private=False  # Đặt là True nếu đây là dữ liệu y tế bảo mật, False nếu muốn công khai (Public)
)
print("Upload completed successfully!")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading dataset to Hugging Face Hub: rimine/Cardiology_ready_dataset...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload completed successfully!


## Lấy danh sách các file NPY

In [33]:
ds = load_dataset("rimine/Cardiology_ready_dataset", split="train")

README.md:   0%|          | 0.00/378 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/334k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [39]:
from huggingface_hub import list_repo_files

repo_id = "UngLong/openm3chest-npy-v2"

# Lấy toàn bộ danh sách file có trong repo
all_files = list_repo_files(repo_id=repo_id, repo_type="dataset")

# Lọc lại để chỉ lấy các file có đuôi .npy (bỏ qua các file cấu hình như .gitattributes, README.md,...)
npy_remote_paths = set([f for f in all_files if f.endswith('.npy')])

print(f"Tìm thấy tổng cộng {len(npy_remote_paths)} file .npy trên Hub.")
# Ép kiểu sang list để có thể lấy 3 phần tử đầu tiên bằng [:3]
print("Ví dụ 3 file đầu tiên:", list(npy_remote_paths)[:3])

Tìm thấy tổng cộng 2951 file .npy trên Hub.
Ví dụ 3 file đầu tiên: ['106427/1.2.840.113654.2.55.310464605003758521494178790294269885159.npy', '117777/1.2.840.113654.2.55.207612969275110567142642639983322290922.npy', '120641/1.2.840.113654.2.55.21749597626029181828941863327425417122.npy']


In [46]:
for i in range(0, 1500):
    t = f"{ds['pids'][i]}/{ds['keys'][i]}"
    if t.strip() in npy_remote_paths:
        print("YES")

In [38]:
# 2. Lọc trực tiếp trên Hugging Face Dataset dùng hàm lambda
# Ghép example['pids'] và example['keys'] của từng dòng thành định dạng "pids/keys" để check
ds_filtered = ds.filter(
    lambda example: f"{str(example['pids']).strip()}/{str(example['keys']).strip()}" in npy_remote_paths
)

print(f"Sau khi lọc, còn lại: {len(ds_filtered)} ca trùng khớp hoàn toàn.")

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Sau khi lọc, còn lại: 0 ca trùng khớp hoàn toàn.
